# Multi-Agent Communication and Coordination

> **The story.** Multi-agent systems help when work has real ownership boundaries, parallel waits, or different authority scopes. Splitting one confused generalist into several confused agents only distributes ambiguity.
>
> **Where you are.** OrderFlow's generalist owns 30 tools, exceeds its context budget, and waits sequentially for supplier and inventory checks.
>
> **Notation.** $A_i$ is specialist agent $i$; $m$ is a typed message envelope; $task_id$ and $correlation_id$ bind delegated work to one lifecycle; $B_i$ is agent $i$'s context budget.

## 0 - The Challenge

> **The mission:** keep every specialist at or below 80% context occupancy, give every delegated task a lifecycle and correlation ID, and reduce modeled completion latency without reducing task success.

```mermaid
flowchart LR
    G["30-tool generalist"] --> O["Context overflow + blocking"]
    O --> D["Typed delegation"]
    D --> S["Supervisor + specialists"]
    S --> M["Measured parallel completion"]
    style G fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, field
from enum import Enum
from typing import Any

from pydantic import BaseModel, Field
from shared import INVENTORY, SUPPLIER_QUOTES, approval_route, estimate_tokens, request_by_id

request = request_by_id("PO-7308")
AGENT_BUDGET = 180
print("Walking incident:", request["email"])


## 1 - Failure First: One Generalist Owns Everything

Tool descriptions and accumulated observations occupy context before the request itself. Sequential waits also add inventory and supplier latency.

```mermaid
flowchart TD
    G["Generalist"] --> T["30 tool descriptions"]
    G --> H["Full history"]
    G --> I["Inventory wait"]
    I --> Q["Supplier wait"]
    T --> O["Budget overflow"]
    H --> O
    style G fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Measure the single-agent baseline ------------------------------------
tool_descriptions = [f"tool_{index}: accepts schema and returns observation" for index in range(30)]
generalist_context = json.dumps({"request": request, "tools": tool_descriptions, "history": ["supplier discussion"] * 10})
generalist_tokens = estimate_tokens(generalist_context)
sequential_latency_ms = 40 + max(q["delay_ms"] for q in SUPPLIER_QUOTES[request["sku"]]) + 20
print(f"Generalist context occupancy: {generalist_tokens / AGENT_BUDGET:.0%}")
print(f"Modeled sequential latency: {sequential_latency_ms} ms")
assert generalist_tokens > AGENT_BUDGET * 0.8


## 2 - Typed Message Envelopes and Task Lifecycle

Natural-language content may vary; the envelope cannot. Sender, recipient, authority scope, task ID, correlation ID, and lifecycle state are deterministic fields.

```mermaid
stateDiagram-v2
    [*] --> submitted
    submitted --> working
    working --> completed
    working --> failed
    failed --> compensated
```


In [ ]:
# -- Define the coordination contract -------------------------------------
class TaskStatus(str, Enum):
    SUBMITTED = "submitted"
    WORKING = "working"
    COMPLETED = "completed"
    FAILED = "failed"
    COMPENSATED = "compensated"

class MessageEnvelope(BaseModel):
    task_id: str
    correlation_id: str
    sender: str
    recipient: str
    authority_scope: set[str]
    message_type: str
    payload: dict[str, Any]
    status: TaskStatus = TaskStatus.SUBMITTED

message = MessageEnvelope(
    task_id="task-7308-inventory",
    correlation_id="corr-7308",
    sender="supervisor",
    recipient="inventory_agent",
    authority_scope={"inventory:read"},
    message_type="inventory.check",
    payload={"sku": request["sku"], "quantity": request["quantity"]},
)
assert message.status == TaskStatus.SUBMITTED and "purchase:approve" not in message.authority_scope
print(message.model_dump_json(indent=2))


## 3 - Router versus Supervisor-Worker

![A no-approval supervisor sends correlated inventory and supplier tasks to scoped specialists in parallel, joins their results at a reducer, and passes the combined evidence to the only specialist allowed to approve](../images/ch08-supervisor-specialists.png)

A router chooses one owner and exits. A supervisor manages dependencies, parallel work, and synthesis. OrderFlow needs the supervisor pattern because inventory and supplier checks can run concurrently before Finance acts.

```mermaid
flowchart TD
    S["Supervisor"] --> I["Inventory specialist"]
    S --> P["Supplier specialist"]
    I --> J["Fan-in"]
    P --> J
    J --> F["Finance specialist"]
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Implement scoped specialists and supervisor fan-out ----------------
def inventory_agent(envelope):
    assert envelope.authority_scope == {"inventory:read"}
    item = INVENTORY[envelope.payload["sku"]]
    return {"task_id": envelope.task_id, "status": "completed", "available": item["available"], "latency_ms": 40}


def supplier_agent(envelope):
    assert envelope.authority_scope == {"supplier:read"}
    quotes = [q for q in SUPPLIER_QUOTES[envelope.payload["sku"]] if q["trusted"] and q["age_hours"] <= 48]
    best = min(quotes, key=lambda q: q["unit_price"])
    return {"task_id": envelope.task_id, "status": "completed", "quote": best, "latency_ms": best["delay_ms"]}


def supervisor(request):
    correlation_id = f"corr-{request['request_id']}"
    inventory_message = MessageEnvelope(task_id=f"{request['request_id']}-inventory", correlation_id=correlation_id, sender="supervisor", recipient="inventory_agent", authority_scope={"inventory:read"}, message_type="inventory.check", payload=request)
    supplier_message = MessageEnvelope(task_id=f"{request['request_id']}-supplier", correlation_id=correlation_id, sender="supervisor", recipient="supplier_agent", authority_scope={"supplier:read"}, message_type="supplier.quote", payload=request)
    with ThreadPoolExecutor(max_workers=2) as executor:
        inventory_future = executor.submit(inventory_agent, inventory_message)
        supplier_future = executor.submit(supplier_agent, supplier_message)
        inventory_result = inventory_future.result()
        supplier_result = supplier_future.result()
    total = supplier_result["quote"]["unit_price"] * request["quantity"]
    return {
        "correlation_id": correlation_id,
        "tasks": [inventory_result, supplier_result],
        "route": approval_route(total),
        "success": True,
        "latency_ms": max(inventory_result["latency_ms"], supplier_result["latency_ms"]) + 20,
    }

multi = supervisor(request)
print(json.dumps(multi, indent=2, default=str))
assert all(task["status"] == "completed" for task in multi["tasks"])
assert len({task["task_id"] for task in multi["tasks"]}) == 2


## 4 - Context Partitioning, Conflict Handling, and A2A Discovery

Each specialist sees its request slice and tools, not the global transcript. Agent cards advertise capabilities; the supervisor still validates authority and resolves conflicting results.

```mermaid
flowchart LR
    C["Agent cards"] --> D["Capability discovery"]
    D --> H["Typed handoff"]
    H --> X{ "Conflicting results?" }
    X -->|"No"| S["Synthesize"]
    X -->|"Yes"| E["Escalate"]
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Measure specialist context and authority separation -----------------
agent_cards = {
    "inventory_agent": {"capabilities": ["inventory.check"], "scopes": ["inventory:read"]},
    "supplier_agent": {"capabilities": ["supplier.quote"], "scopes": ["supplier:read"]},
    "finance_agent": {"capabilities": ["purchase.approve"], "scopes": ["purchase:approve"]},
}
contexts = {
    "inventory_agent": json.dumps({"request": {"sku": request["sku"], "quantity": request["quantity"]}, "tools": agent_cards["inventory_agent"]}),
    "supplier_agent": json.dumps({"request": {"sku": request["sku"]}, "tools": agent_cards["supplier_agent"]}),
    "finance_agent": json.dumps({"request": {"request_id": request["request_id"], "budget": request["budget"]}, "route": multi["route"], "tools": agent_cards["finance_agent"]}),
}
occupancies = {agent: estimate_tokens(context) / AGENT_BUDGET for agent, context in contexts.items()}
print("Context occupancy:", {agent: f"{value:.0%}" for agent, value in occupancies.items()})
assert all(value <= 0.80 for value in occupancies.values())
assert "purchase:approve" not in agent_cards["inventory_agent"]["scopes"]
assert "purchase:approve" not in agent_cards["supplier_agent"]["scopes"]


In [ ]:
# -- Compare single and multi-agent outcomes ------------------------------
single = {"success": True, "latency_ms": sequential_latency_ms, "context_tokens": generalist_tokens}
print("Single-agent baseline:", single)
print("Supervisor-worker:", {"success": multi["success"], "latency_ms": multi["latency_ms"], "max_context_occupancy": max(occupancies.values())})
assert multi["success"] == single["success"]
assert multi["latency_ms"] < single["latency_ms"]
assert max(occupancies.values()) <= 0.80
print("PASS: parallel completion improved without reducing task success or expanding authority.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Coordination targets met"] --> B["Next: recovery"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Single generalist | Supervisor-worker |
|---|---:|---:|
| Maximum context occupancy | Above 80% | At or below 80% |
| Supplier + inventory latency | Sequential sum | Parallel maximum |
| Delegated lifecycle | Implicit | Task IDs and statuses |
| Specialist approval authority | Ambiguous | Inventory and supplier cannot approve |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Typed envelopes, lifecycle, supervisor, fan-out/fan-in, context partitioning, agent cards |
| Explained and illustrated | Router and peer patterns, conflict handling, shared blackboard |
| Named with a reason | Remote A2A transport, deferred because local envelope semantics come first |

### Key Takeaways

- Split agents at ownership and authority boundaries, not by job-title aesthetics.
- Flexible content still needs a rigid envelope.
- A supervisor earns its cost when dependencies and parallel waits exist.
- Specialists receive less context and less authority, not merely different prompts.
